In [1]:
from ast import literal_eval
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import wandb
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score, accuracy_score

In [2]:
generator = torch.random.manual_seed(42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
dataset_df = pd.read_csv('/kaggle/input/lstm-challenge3/data_train_preprocessing.csv', converters={'token': literal_eval})
dataset_df = dataset_df[['token', 'label']]

In [5]:
vocab = set()

max_length = 0

for tokens in dataset_df['token']:
    vocab.update(tokens)
    max_length = max(max_length, len(tokens))

# Add padding token
vocab.add('<pad>')

word2idx = {word: idx for idx, word in enumerate(vocab)}


def prepare_sequence(seq, to_ix, padding):
    pad = np.zeros((padding,), dtype=np.int64)
    pad[-len(seq):] = [to_ix[word] for word in seq]
    return torch.from_numpy(pad)


class TokenDataset(Dataset):
    def __init__(self, dataframe):
        data = [prepare_sequence(tokens, word2idx, max_length) for tokens in dataframe['token']]
        labels = dataframe['label'].tolist()

        self.data = torch.stack(data)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

In [6]:
batch_size = 32
# EMBEDDING_DIM = 50
HIDDEN_DIM = 64

dataset = TokenDataset(dataset_df)

training_set, validation_set, test_set = torch.utils.data.random_split(dataset, [0.7, 0.15, 0.15], generator=generator)

train_loader = DataLoader(training_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(validation_set, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

In [7]:
class LSTMSentiment(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, num_layers):
        super(LSTMSentiment, self).__init__()

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.num_layers = num_layers

        self.embedding = nn.Embedding(self.vocab_size, self.embedding_dim)
        self.lstm = nn.LSTM(self.embedding_dim, self.hidden_dim, self.num_layers, batch_first=True)
        self.fc = nn.Linear(self.hidden_dim, self.output_dim)

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.lstm(x)
        x = self.fc(x[:, -1, :])
        return x

    def init_hiddden(self, batch_size):
        h0 = torch.zeros(1, batch_size, self.hidden_dim).to(device)
        c0 = torch.zeros(1, batch_size, self.hidden_dim).to(device)
        return h0, c0

In [8]:
def train_one_epoch(model, data_loader, optimizer, criterion, scheduler):
    model.train()
    total_loss = 0

    for tokens, labels in data_loader:
        tokens, labels = tokens.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(tokens)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    scheduler.step()
    return total_loss / len(data_loader)


def validate(model, data_loader, criterion):
    model.eval()
    total_loss = 0
    predictions = []
    true_labels = []

    with torch.no_grad():
        for tokens, labels in data_loader:
            tokens, labels = tokens.to(device), labels.to(device)

            outputs = model(tokens)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            predictions.extend(predicted.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions, average='macro')
    return total_loss / len(data_loader), accuracy, f1

In [9]:
def train(lr, step_size, epochs, num_layers, embedding_dim, use_wandb=False):
    model = LSTMSentiment(len(word2idx), embedding_dim, HIDDEN_DIM, 3, num_layers).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size)

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scheduler)
        val_loss, val_accuracy, val_f1 = validate(model, val_loader, criterion)
        test_loss, test_accuracy, test_f1 = validate(model, test_loader, criterion)

        print(f'Epoch {epoch + 1}/{epochs}, '
              f'Train Loss: {train_loss:.2f}, '
              f'Val Loss: {val_loss:.2f}, Val Accuracy: {val_accuracy:.2f}, Val F1: {val_f1:.2f}, '
              f'Test Loss: {test_loss:.2f}, Test Accuracy: {test_accuracy:.2f}, Test F1: {test_f1:.2f}')
        if use_wandb:
            wandb.log({
                'epoch': epoch + 1,
                'train_loss': train_loss,
                'val_loss': val_loss,
                'val_accuracy': val_accuracy,
                'val_f1': val_f1,
                'test_loss': test_loss,
                'test_accuracy': test_accuracy,
                'test_f1': test_f1
            })

In [10]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
key = user_secrets.get_secret('wandb-api-key')

wandb.login(key=key)

sweep_configuration = {
    "method": "grid",
    "metric": {"goal": "maximize", "name": "val_f1"},
    'name': "LSTM sweep",
    "parameters": {
        "lr": {'values': [1e-2, 1e-3, 1e-4]},
        "step_size": {'values': [3, 5, 7, 10]},
        "num_layers": {'values': [1, 2, 3, 5]},
        "embedding_dim": {'values': [16, 64, 128, 256]},
    },
}


def train_wrapper():
    with wandb.init() as run:
        train(
            lr=run.config.lr,
            step_size=run.config.step_size,
            embedding_dim=run.config.embedding_dim,
            epochs=5,
            num_layers=run.config.num_layers,
            use_wandb=True
        )


sweep_id = wandb.sweep(sweep=sweep_configuration, entity='matteo-ghia-politecnico-di-torino', project="aml challenge 3")
print(f"Sweep ID: {sweep_id}")
wandb.agent(sweep_id, function=train_wrapper)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: matteo-ghia (matteo-ghia-2001) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Create sweep with ID: gf6sc2kt
Sweep URL: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%203/sweeps/gf6sc2kt
Sweep ID: gf6sc2kt


wandb: Agent Starting Run: 91nt5l9q with config:
wandb: 	embedding_dim: 16
wandb: 	lr: 0.01
wandb: 	num_layers: 1
wandb: 	step_size: 3
wandb: Currently logged in as: matteo-ghia (matteo-ghia-politecnico-di-torino) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.19.9
wandb: Run data is saved locally in /kaggle/working/wandb/run-20250603_080151-91nt5l9q
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run different-sweep-1
wandb: ⭐️ View project at https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%203
wandb: 🧹 View sweep at https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%203/sweeps/gf6sc2kt
wandb: 🚀 View run at https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%203/runs/91nt5l9q


Epoch 1/5, Train Loss: 0.88, Val Loss: 0.78, Val Accuracy: 0.67, Val F1: 0.67, Test Loss: 0.75, Test Accuracy: 0.68, Test F1: 0.68
Epoch 2/5, Train Loss: 0.66, Val Loss: 0.76, Val Accuracy: 0.69, Val F1: 0.69, Test Loss: 0.73, Test Accuracy: 0.69, Test F1: 0.70
Epoch 3/5, Train Loss: 0.54, Val Loss: 0.78, Val Accuracy: 0.69, Val F1: 0.69, Test Loss: 0.75, Test Accuracy: 0.69, Test F1: 0.70
Epoch 4/5, Train Loss: 0.38, Val Loss: 0.85, Val Accuracy: 0.68, Val F1: 0.68, Test Loss: 0.81, Test Accuracy: 0.69, Test F1: 0.70
Epoch 5/5, Train Loss: 0.33, Val Loss: 0.91, Val Accuracy: 0.67, Val F1: 0.68, Test Loss: 0.87, Test Accuracy: 0.69, Test F1: 0.69


wandb:                                                                                
wandb: 
wandb: Run history:
wandb:         epoch ▁▃▅▆█
wandb: test_accuracy ▁███▆
wandb:       test_f1 ▁███▇
wandb:     test_loss ▂▁▂▅█
wandb:    train_loss █▅▄▂▁
wandb:  val_accuracy ▁██▅▄
wandb:        val_f1 ▁██▅▄
wandb:      val_loss ▂▁▂▅█
wandb: 
wandb: Run summary:
wandb:         epoch 5
wandb: test_accuracy 0.6922
wandb:       test_f1 0.69421
wandb:     test_loss 0.8733
wandb:    train_loss 0.33006
wandb:  val_accuracy 0.67488
wandb:        val_f1 0.67784
wandb:      val_loss 0.90952
wandb: 
wandb: 🚀 View run different-sweep-1 at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%203/runs/91nt5l9q
wandb: ⭐️ View project at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%203
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20250603_080151-91nt5l9q/logs
wandb: Agent Starting Run: 5evu8clm wi

...

wandb:                                                                                
wandb: 
wandb: Run history:
wandb:         epoch ▁▃▅▆█
wandb: test_accuracy ▁▅▆▇█
wandb:       test_f1 ▁▄▆▇█
wandb:     test_loss █▅▃▂▁
wandb:    train_loss █▇▄▃▁
wandb:  val_accuracy ▁▄▆▇█
wandb:        val_f1 ▁▄▆██
wandb:      val_loss █▅▃▂▁
wandb: 
wandb: Run summary:
wandb:         epoch 5
wandb: test_accuracy 0.63319
wandb:       test_f1 0.62593
wandb:     test_loss 0.81422
wandb:    train_loss 0.77441
wandb:  val_accuracy 0.62317
wandb:        val_f1 0.61845
wandb:      val_loss 0.82736
wandb: 
wandb: 🚀 View run major-sweep-190 at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%203/runs/3w1zex6m
wandb: ⭐️ View project at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%203
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20250603_124456-3w1zex6m/logs
wandb: Agent Starting Run: z9yjyyf5 wi

Epoch 1/5, Train Loss: 1.09, Val Loss: 1.08, Val Accuracy: 0.40, Val F1: 0.19, Test Loss: 1.07, Test Accuracy: 0.41, Test F1: 0.20
Epoch 2/5, Train Loss: 1.03, Val Loss: 0.95, Val Accuracy: 0.53, Val F1: 0.53, Test Loss: 0.93, Test Accuracy: 0.55, Test F1: 0.55
Epoch 3/5, Train Loss: 0.87, Val Loss: 0.88, Val Accuracy: 0.59, Val F1: 0.57, Test Loss: 0.85, Test Accuracy: 0.61, Test F1: 0.60
Epoch 4/5, Train Loss: 0.78, Val Loss: 0.85, Val Accuracy: 0.61, Val F1: 0.61, Test Loss: 0.82, Test Accuracy: 0.64, Test F1: 0.64
Epoch 5/5, Train Loss: 0.72, Val Loss: 0.82, Val Accuracy: 0.64, Val F1: 0.64, Test Loss: 0.79, Test Accuracy: 0.66, Test F1: 0.65


wandb:                                                                                
wandb: 
wandb: Run history:
wandb:         epoch ▁▃▅▆█
wandb: test_accuracy ▁▅▇▇█
wandb:       test_f1 ▁▆▇██
wandb:     test_loss █▄▂▂▁
wandb:    train_loss █▇▄▂▁
wandb:  val_accuracy ▁▅▇▇█
wandb:        val_f1 ▁▆▇██
wandb:      val_loss █▄▃▂▁
wandb: 
wandb: Run summary:
wandb:         epoch 5
wandb: test_accuracy 0.65647
wandb:       test_f1 0.65402
wandb:     test_loss 0.79138
wandb:    train_loss 0.72223
wandb:  val_accuracy 0.63617
wandb:        val_f1 0.6359
wandb:      val_loss 0.81606
wandb: 
wandb: 🚀 View run dark-sweep-191 at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%203/runs/z9yjyyf5
wandb: ⭐️ View project at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%203
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20250603_124718-z9yjyyf5/logs
wandb: Agent Starting Run: fob22vu0 with

Epoch 1/5, Train Loss: 1.09, Val Loss: 1.08, Val Accuracy: 0.41, Val F1: 0.22, Test Loss: 1.07, Test Accuracy: 0.42, Test F1: 0.22
Epoch 2/5, Train Loss: 1.01, Val Loss: 0.94, Val Accuracy: 0.53, Val F1: 0.45, Test Loss: 0.93, Test Accuracy: 0.54, Test F1: 0.46
Epoch 3/5, Train Loss: 0.88, Val Loss: 0.89, Val Accuracy: 0.58, Val F1: 0.58, Test Loss: 0.87, Test Accuracy: 0.60, Test F1: 0.59
Epoch 4/5, Train Loss: 0.81, Val Loss: 0.84, Val Accuracy: 0.62, Val F1: 0.61, Test Loss: 0.82, Test Accuracy: 0.64, Test F1: 0.63
Epoch 5/5, Train Loss: 0.74, Val Loss: 0.82, Val Accuracy: 0.64, Val F1: 0.64, Test Loss: 0.79, Test Accuracy: 0.65, Test F1: 0.65


wandb:                                                                                
wandb: 
wandb: Run history:
wandb:         epoch ▁▃▅▆█
wandb: test_accuracy ▁▅▆▇█
wandb:       test_f1 ▁▅▇██
wandb:     test_loss █▄▃▂▁
wandb:    train_loss █▆▄▂▁
wandb:  val_accuracy ▁▅▆▇█
wandb:        val_f1 ▁▅▇██
wandb:      val_loss █▄▃▂▁
wandb: 
wandb: Run summary:
wandb:         epoch 5
wandb: test_accuracy 0.65457
wandb:       test_f1 0.64976
wandb:     test_loss 0.79061
wandb:    train_loss 0.742
wandb:  val_accuracy 0.64266
wandb:        val_f1 0.64046
wandb:      val_loss 0.81735
wandb: 
wandb: 🚀 View run dainty-sweep-192 at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%203/runs/fob22vu0
wandb: ⭐️ View project at: https://wandb.ai/matteo-ghia-politecnico-di-torino/aml%20challenge%203
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20250603_125041-fob22vu0/logs
wandb: Sweep Agent: Waiting for job.
wa

In [11]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# key = user_secrets.get_secret('wandb-api-key')

# wandb.login(key=key)

# with wandb.init(project="aml challenge 3", entity="matteo-ghia-politecnico-di-torino", name='Conv1D autoencoder') as run:
# model = train(lr=1e-2, step_size=10, embedding_dim=64, epochs=10, num_layers=4, use_wandb=False)
# torch.save(model.state_dict(), "model.pt")